In [ ]:
from utils import *
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
import json
from loaders import baseline_loader

In [ ]:
res_df, scalars = baseline_loader()

In [ ]:
envs = res_df["env"].unique()

fig = make_subplots(
    rows=1,
    cols=len(envs),
    column_titles=[*envs],
)

for col, env in enumerate(envs, 1):
    dfs = []
    for _, test in res_df[res_df["env"] == env].iterrows():
        df = scalars.read(test["path"])
        df = df[df["tag"] == "val/mean_ep_ret"]
        df["index"] = np.arange(len(df))
        df["ratio"] = test["ratio"]
        dfs.append(df)
    df = pd.concat(dfs)

    g = df.groupby(["index", "ratio"])
    avg_df = pd.DataFrame.from_records(
        {
            "score_mean": g["value"].mean(),
            "score_std": g["value"].std(),
            "step": g["step"].median(),
        }
    )
    avg_df = avg_df.reset_index()

    colors = make_color_iter(palette="Dark24")
    for ratio, color in zip(avg_df["ratio"].unique(), colors):
        df = avg_df[avg_df["ratio"] == ratio]

        kw = dict(name=f"Ratio = {ratio}")
        if col != 1:
            kw.update(showlegend=False)

        for trace in err_line(
            x=df["step"],
            y=df["score_mean"],
            std=df["score_std"],
            color=color,
            **kw,
        ):
            fig.add_trace(trace, row=1, col=col)

fig

In [ ]:
final_df = []
for _, test in res_df.iterrows():
    df = scalars.read(test["path"])
    df = df[df["tag"] == "val/mean_ep_ret"]
    last = df.iloc[-1]["value"]
    final_df.append({"path": test["path"], "score": last})
final_df = pd.DataFrame.from_records(final_df)
final_df = pd.merge(final_df, res_df, on="path")
final_df

In [ ]:
with open("ref_scores/baselines.json", "rb") as f:
    baselines = json.load(f)

records = []
for task in baselines:
    name = task.removeprefix("atari_")
    name = "".join(w.capitalize() for w in name.split("_"))
    records.append(
        {
            "task": name,
            **{
                k: baselines[task].get(k)
                for k in ("random", "human_gamer", "human_record")
            },
        }
    )

base_df = pd.DataFrame.from_records(records)
base_df

In [ ]:
envs = final_df["env"].unique()

fig = make_subplots(
    rows=1,
    cols=len(envs),
    column_titles=[*envs],
)

for col, env in enumerate(envs, 1):
    df_ = final_df[final_df["env"] == env]

    g = df_.groupby("ratio")
    avg_df = pd.DataFrame.from_records(
        {
            "score_mean": g["score"].mean(),
            "score_std": g["score"].std(),
            "ratio": g["ratio"].first(),
        }
    )

    color = next(make_color_iter())
    for trace in err_line(
        x=avg_df["ratio"],
        y=avg_df["score_mean"],
        std=avg_df["score_std"],
        color=color,
        showlegend=False,
    ):
        fig.add_trace(trace, row=1, col=col)

    fig.update_xaxes(type="category", row=1, col=col)

    rand = base_df[base_df["task"] == env]["random"].item()
    fig.update_yaxes(range=[rand, None])

fig.update_layout(width=1000, height=400)

fig.write_image("../tex/assets/baseline_score.pdf")
fig

In [ ]:
envs = final_df["env"].unique()

fig = make_subplots(
    rows=1,
    cols=len(envs),
    column_titles=[*envs],
)

for col, env in enumerate(envs, 1):
    score_df = final_df[final_df["env"] == env].copy()
    score_df["wm_loss"] = pd.Series(dtype=np.float32)

    for idx, row in score_df.iterrows():
        test_df = scalars.read(row["path"])
        test_df = test_df[test_df["tag"] == "val/wm_loss"]
        score_df.at[idx, "wm_loss"] = test_df.iloc[-1]["value"]

    subplot = px.scatter(score_df, x="score", y="wm_loss", trendline="ols")
    for trace in subplot.data:
        fig.add_trace(trace, row=1, col=col)

fig.update_layout(width=1000, height=400)

fig.write_image("../tex/assets/baseline_model_val.pdf")
fig

In [ ]:
envs = final_df["env"].unique()

fig = make_subplots(
    rows=1,
    cols=len(envs),
    column_titles=[*envs],
)

for col, env in enumerate(envs, 1):
    score_df = final_df[final_df["env"] == env].copy()
    score_df["wm_loss"] = pd.Series(dtype=np.float32)

    for idx, row in score_df.iterrows():
        test_df = scalars.read(row["path"])
        test_df = test_df[test_df["tag"] == "wm/loss"]
        score_df.at[idx, "wm_loss"] = test_df.iloc[-1]["value"]

    subplot = px.scatter(score_df, x="score", y="wm_loss", trendline="ols")
    for trace in subplot.data:
        fig.add_trace(trace, row=1, col=col)

fig.update_layout(width=1000, height=400)

fig.write_image("../tex/assets/baseline_model_train.pdf")
fig